# Machine Learning Module - Group Project Assignment
## Project Title: Universal Business Review Analyzer & Sentiment Intelligence System
### Domain: Multi-Industry Customer Feedback Analytics (Retail, Hospitality, Tech, Services)

---
### Assignment Compliance Matrix
| Assignment Requirement | Implementation in this Notebook |
| :--- | :--- |
| **1. Problem Selection** | Multi-domain business review analyzer for universal sentiment classification across diverse industries. |
| **2. 100,000 Dataset & EDA** | Large-scale 100,000 record multi-domain dataset covering Retail, Hospitality, Consumer Tech, and Services. |
| **3. Feature Engineering (Mandatory)** | **6 Meaningful Techniques**: Text cleaning, N-gram TF-IDF, metadata features, emotional punctuation signals, lexicon polarity, standard scaling. |
| **4. Multi-Model Development** | Comparison across candidate models including Logistic Regression, LinearSVC, Random Forest, and PyTorch Deep Neural Network. |
| **5. Model Evaluation** | Accuracy, Precision, Recall, F1-Score, 5-Fold Cross Validation, Confusion Matrix, Classification Report. |
| **6. Model Serialization** | Trained model checkpoint exported as PyTorch `.pt` format and `.joblib` pipeline bundle for REST API deployment. |

## Step 1: Environment and Dependencies Setup
Initialize libraries for NLP, feature engineering, machine learning, and deep learning.

In [2]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
os.makedirs("models", exist_ok=True)
print("[INFO] All required libraries loaded successfully.")

[INFO] All required libraries loaded successfully.


## Step 2: 100,000-Record Dataset Ingestion and Exploratory Data Analysis (EDA)
Load the multi-domain business reviews dataset from `data/universal_business_reviews_100k.csv`.

In [4]:
data_path = "data/universal_business_reviews_100k.csv"

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found at {data_path}. Please generate the 100k dataset first.")

df_full = pd.read_csv(data_path)

print("=" * 60)
print(f"Total Records in Dataset : {len(df_full):,}")
print(f"Feature Columns          : {list(df_full.columns)}")
print(f"Missing Values Count     :\n{df_full.isnull().sum()}")
print(f"Duplicate Review Records : {df_full.duplicated(subset=['review_text']).sum()}")
print("=" * 60)

# Clean dataset
df_clean = df_full.dropna(subset=['review_text']).drop_duplicates(subset=['review_text']).reset_index(drop=True)

# Sample for memory-efficient training
SAMPLE_SIZE = 25000
df = df_clean.sample(n=min(SAMPLE_SIZE, len(df_clean)), random_state=RANDOM_STATE).reset_index(drop=True)

print(f"[INFO] Cleaned Working Sample: {len(df):,} records.")
print("\nTarget Class Balance:")
print(df['sentiment'].value_counts(normalize=True).rename({1: 'Positive (1)', 0: 'Negative (0)'}))

df.head()

Total Records in Dataset : 100,000
Feature Columns          : ['review_id', 'review_text', 'rating', 'sentiment', 'category', 'verified_purchase']
Missing Values Count     :
review_id            0
review_text          0
rating               0
sentiment            0
category             0
verified_purchase    0
dtype: int64
Duplicate Review Records : 56117
[INFO] Cleaned Working Sample: 25,000 records.

Target Class Balance:
sentiment
Positive (1)    0.56568
Negative (0)    0.43432
Name: proportion, dtype: float64


,review_id,review_text,rating,sentiment,category,verified_purchase
0,REV-030050,Such a huge letdown regarding the truffle past...,1,0,Hospitality & Food,True
1,REV-059363,Incredible performance and reliability from th...,5,1,Consumer Tech & Electronics,False
2,REV-035963,A five star experience from start to finish wi...,5,1,Hospitality & Food,True
3,REV-027176,Such a huge letdown regarding the craft coffee...,1,0,Hospitality & Food,True
4,REV-099193,Really happy with the car detailing. It offers...,4,1,Services & Professional,False
